## Project Validation Script

In [1]:
import os
import numpy as np
import pandas as pd
from configparser import ConfigParser
from ConnectionPool import pool
import psycopg2


['image_paths', 'model_paths', 'error', 'find_shift', 'database']
Connection pool created successfully


In [2]:
config = ConfigParser()
config.read('./config.ini')

small_img_pth = config['image_paths']['total_zoom_small_path']
large_img_pth = config['image_paths']['total_zoom_large_path']
obd_images_path = config['image_paths']['obj_detect_img_path']

print("=== CONFIG CHECK ===")
print("Small images path:", small_img_pth)
print("Large images path:", large_img_pth)
print("OBD images path:", obd_images_path)
print()

=== CONFIG CHECK ===
Small images path: ./images_small
Large images path: ./images_large
OBD images path: ./obj_detect_images



In [3]:
def list_image_files(directory):
    if not os.path.isdir(directory):
        print(f"[WARN] Directory does not exist: {directory}")
        return []
    exts = ('.png', '.jpg', '.jpeg')
    return sorted([f for f in os.listdir(directory) if f.lower().endswith(exts)])

def parse_ids_from_filename(filename):
    """
    Expected pattern (no extension):
      nge_object_id_field_search_id_sport_name
    Returns (field_search_id, nge_object_id, sport_name)
    """
    base = os.path.splitext(filename)[0]
    parts = base.split("_")
    nge_object_id = None
    field_search_id = None
    sport_name = None

    if len(parts) >= 2:
        try:
            nge_object_id = int(parts[0])
        except ValueError:
            pass
        try:
            field_search_id = int(parts[1])
        except ValueError:
            pass
    if len(parts) >= 3:
        sport_name = "_".join(parts[2:])
    return field_search_id, nge_object_id, sport_name

def connect_db():
    try:
        conn = pool.getconn()
        print("DB connection OK.")
        return conn
    except Exception as e:
        print("DB connection FAILED:", e)
        return None

In [4]:
print("=== IMAGE NAMING & COUNT CHECKS ===")

small_files = list_image_files(small_img_pth)
large_files = list_image_files(large_img_pth)

print(f"Small images: {len(small_files)}")
print(f"Large images: {len(large_files)}")

small_ids = set(os.path.splitext(f)[0] for f in small_files)
large_ids = set(os.path.splitext(f)[0] for f in large_files)

missing_in_large = small_ids - large_ids
missing_in_small = large_ids - small_ids

print(f"Missing in large (present only in small): {len(missing_in_large)}")
print(f"Missing in small (present only in large): {len(missing_in_small)}")

if missing_in_large:
    print("Examples missing in large:", list(missing_in_large)[:10])
if missing_in_small:
    print("Examples missing in small:", list(missing_in_small)[:10])

bad_name_examples = []
parsed_records = []

for f in small_files:
    field_id, obj_id, sport_name = parse_ids_from_filename(f)
    if field_id is None or obj_id is None:
        bad_name_examples.append(f)
    parsed_records.append((f, field_id, obj_id, sport_name))

print(f"Small images with invalid naming pattern: {len(bad_name_examples)}")
if bad_name_examples:
    print("Examples:", bad_name_examples[:10])

img_df = pd.DataFrame(parsed_records,
                      columns=["filename", "field_search_id", "nge_object_id", "sport_name"])
print("\nParsed image naming sample:")
print(img_df.head())

dup_pairs = img_df.groupby(['field_search_id', 'nge_object_id']).size().reset_index(name='count')
dup_pairs = dup_pairs[dup_pairs['count'] > 1]
print(f"\nDuplicate image (field_search_id, nge_object_id) pairs: {len(dup_pairs)}")
if not dup_pairs.empty:
    print(dup_pairs.head())

=== IMAGE NAMING & COUNT CHECKS ===
Small images: 332
Large images: 332
Missing in large (present only in small): 0
Missing in small (present only in large): 0
Small images with invalid naming pattern: 332
Examples: ['1000.0_9802_Tennis.png', '1001.0_9802_Tennis.png', '1002.0_9802_Tennis.png', '1003.0_9802_Tennis.png', '1004.0_9802_Tennis.png', '1005.0_9803_Tennis.png', '1006.0_9803_Stadium.png', '1007.0_9809_Tennis.png', '1008.0_9809_Tennis.png', '1009.0_9809_Tennis.png']

Parsed image naming sample:
                 filename  field_search_id nge_object_id sport_name
0  1000.0_9802_Tennis.png             9802          None     Tennis
1  1001.0_9802_Tennis.png             9802          None     Tennis
2  1002.0_9802_Tennis.png             9802          None     Tennis
3  1003.0_9802_Tennis.png             9802          None     Tennis
4  1004.0_9802_Tennis.png             9802          None     Tennis

Duplicate image (field_search_id, nge_object_id) pairs: 0


In [5]:
print("\n=== EXCEL / MODEL OUTPUT CHECK ===")

excel_files = [f for f in os.listdir('.') if f.startswith('combined_results_') and f.endswith('.xlsx')]
if not excel_files:
    print("[WARN] No combined_results_*.xlsx found.")
    combined_df = pd.DataFrame()
else:
    excel_files.sort(key=lambda x: os.path.getmtime(x), reverse=True)
    latest_excel = excel_files[0]
    print("Using Excel:", latest_excel)
    combined_df = pd.read_excel(latest_excel)

if combined_df.empty:
    print("combined_results is empty, cannot run further validation on model outputs.")
else:
    print("combined_results columns:", list(combined_df.columns))
    print(combined_df.head())

    print("\nRows in combined_results:", len(combined_df))
    print("Unique field_search_id:", combined_df['field_search_id'].nunique())
    print("Unique nge_object_id:", combined_df['nge_object_id'].nunique())

    prob_cols = [
        'predicted_field_type_probability',
        'predicted_sport_type_probability',
        'large_predicted_field_type_probability',
        'large_predicted_sport_type_probability'
    ]
    for col in prob_cols:
        if col in combined_df.columns:
            print(f"\nStats for {col}:")
            print(combined_df[col].describe())
        else:
            print(f"[WARN] Missing probability column in Excel: {col}")

    dup_excel = combined_df.groupby(['field_search_id', 'nge_object_id']).size().reset_index(name='count')
    dup_excel = dup_excel[dup_excel['count'] > 1]
    print("\nDuplicate (field_search_id, nge_object_id) in Excel:", len(dup_excel))
    if not dup_excel.empty:
        print(dup_excel.head())


=== EXCEL / MODEL OUTPUT CHECK ===
Using Excel: combined_results_2026-01-22T10-52-53.xlsx
combined_results columns: ['nge_object_id', 'field_search_id', 'predicted_field_type', 'predicted_sport', 'predicted_field_type_probability', 'predicted_sport_type_probability', 'predicted_large_field_type', 'predicted_large_sport', 'large_predicted_field_type_probability', 'large_predicted_sport_type_probability', 'detected_sport', 'detect_confidence']
   nge_object_id  field_search_id predicted_field_type predicted_sport  \
0            789             9732                 Turf          Soccer   
1            905             9732                 Turf          Soccer   
2            790             9737            Buildings       Buildings   
3            906             9737            Buildings       Buildings   
4            791             9738            Buildings       Buildings   

   predicted_field_type_probability  predicted_sport_type_probability  \
0                          0.999941

In [6]:
print("\n=== DATABASE CONSISTENCY & ACCURACY CHECK ===")

conn = connect_db()
if conn and not combined_df.empty:
    try:
        cur = conn.cursor()

        cur.execute("SELECT COUNT(*) FROM public.new_google_earth;")
        nge_count = cur.fetchone()[0]
        print("Rows in new_google_earth:", nge_count)

        cur.execute("SELECT COUNT(*) FROM public.nge_object;")
        obj_count = cur.fetchone()[0]
        print("Rows in nge_object:", obj_count)

        # ---- 4a. Compare predicted_sport vs nge_object.sport_name ----
        # We assume nge_object has (nge_object_id, field_search_id, sport_name)
        print("\n--- Comparison: predicted_sport vs nge_object.sport_name ---")

        # Limit to reasonably confident predictions (optional)
        if 'predicted_sport_type_probability' in combined_df.columns:
            confident_df = combined_df[
                combined_df['predicted_sport_type_probability'] >= 0.8
            ].copy()
        else:
            confident_df = combined_df.copy()

        print("Rows in confident_df:", len(confident_df))

        # Pull matching rows from DB
        ids = confident_df[['field_search_id', 'nge_object_id']].drop_duplicates()
        if not ids.empty:
            placeholders = ",".join(["(%s,%s)"] * len(ids))
            params = []
            for _, r in ids.iterrows():
                params.extend([int(r['field_search_id']), int(r['nge_object_id'])])

            query = f"""
                SELECT no.nge_object_id,
                       no.field_search_id,
                       no.sport_name
                FROM public.nge_object no
                JOIN (VALUES {placeholders}) AS v(field_search_id, nge_object_id)
                  ON no.field_search_id = v.field_search_id
                 AND no.nge_object_id = v.nge_object_id;
            """
            cur.execute(query, tuple(params))
            db_rows = cur.fetchall()
            db_df = pd.DataFrame(db_rows,
                                 columns=['nge_object_id', 'field_search_id', 'db_sport_name'])

            merged = confident_df.merge(
                db_df,
                on=['field_search_id', 'nge_object_id'],
                how='left'
            )

            # Basic accuracy: predicted_sport == db_sport_name
            valid_rows = merged[merged['db_sport_name'].notna()].copy()
            if not valid_rows.empty:
                valid_rows['correct'] = valid_rows['predicted_sport'] == valid_rows['db_sport_name']
                accuracy = valid_rows['correct'].mean()
                print(f"\nMatched rows for comparison: {len(valid_rows)}")
                print(f"Sport name accuracy vs DB: {accuracy:.3f}")

                mismatches = valid_rows[~valid_rows['correct']]
                print("Mismatches count:", len(mismatches))
                if not mismatches.empty:
                    print("Sample mismatches:")
                    print(mismatches[['field_search_id',
                                      'nge_object_id',
                                      'predicted_sport',
                                      'db_sport_name']].head(10))
            else:
                print("No overlapping IDs between Excel and nge_object to compare sports.")
        else:
            print("No IDs to compare in confident_df.")

        # ---- 4b. Optional: compare high-level field classification vs new_google_earth.object_sport ----
        print("\n--- Comparison: predicted_sport vs new_google_earth.object_sport (high level) ---")

        if 'predicted_sport' in combined_df.columns:
            # Pull distinct field_search_ids
            field_ids = combined_df['field_search_id'].dropna().unique().tolist()
            if field_ids:
                placeholders = ",".join(["%s"] * len(field_ids))
                q2 = f"""
                    SELECT field_search_id,
                           object_sport
                    FROM public.new_google_earth
                    WHERE field_search_id IN ({placeholders});
                """
                cur.execute(q2, tuple(field_ids))
                rows2 = cur.fetchall()
                nge_df = pd.DataFrame(rows2, columns=['field_search_id', 'object_sport'])

                # Aggregate predicted_sport per field (e.g., most frequent)
                agg = (combined_df.groupby('field_search_id')['predicted_sport']
                                  .agg(lambda x: x.value_counts().idxmax())
                                  .reset_index(name='predicted_sport_majority'))

                merged2 = agg.merge(nge_df, on='field_search_id', how='left')
                valid2 = merged2[merged2['object_sport'].notna()].copy()
                if not valid2.empty:
                    valid2['correct'] = valid2['predicted_sport_majority'] == valid2['object_sport']
                    acc2 = valid2['correct'].mean()
                    print(f"Fields with comparable sport labels: {len(valid2)}")
                    print(f"Field-level sport accuracy vs new_google_earth.object_sport: {acc2:.3f}")

                    mism2 = valid2[~valid2['correct']]
                    print("Sample field-level mismatches:")
                    print(mism2[['field_search_id',
                                 'predicted_sport_majority',
                                 'object_sport']].head(10))
                else:
                    print("No comparable records in new_google_earth for field-level check.")
            else:
                print("No field_search_id values in combined_df.")
        else:
            print("predicted_sport column missing in Excel; cannot run field-level comparison.")

    except Exception as e:
        print("Error during DB validation/comparison:", e)
    finally:
        cur.close()
        pool.putconn(conn)
else:
    if not conn:
        print("Skipping DB validation: no DB connection.")
    if combined_df.empty:
        print("Skipping DB validation: combined_results is empty.")


=== DATABASE CONSISTENCY & ACCURACY CHECK ===
DB connection OK.
Rows in new_google_earth: 101
Rows in nge_object: 232

--- Comparison: predicted_sport vs nge_object.sport_name ---
Rows in confident_df: 196

Matched rows for comparison: 196
Sport name accuracy vs DB: 0.684
Mismatches count: 62
Sample mismatches:
    field_search_id  nge_object_id predicted_sport db_sport_name
0              9732            789          Soccer       Stadium
1              9732            905          Soccer       Stadium
2              9738            791       Buildings      Baseball
3              9738            792          Soccer       Stadium
4              9738            907       Buildings      Baseball
5              9738            908          Soccer       Stadium
24             9741            803       Buildings        Tennis
26             9741            919       Buildings        Tennis
28             9744            805      Basketball        Tennis
29             9744            921  

In [7]:
print("\n=== SUMMARY FLAGS ===")
issues = []

if missing_in_large:
    issues.append("Some small images have no matching large image.")
if missing_in_small:
    issues.append("Some large images have no matching small image.")
if not img_df.empty and (img_df['field_search_id'].isna() | img_df['nge_object_id'].isna()).any():
    issues.append("Some image filenames do not parse valid field_search_id/nge_object_id.")
if not combined_df.empty and not dup_excel.empty:
    issues.append("Duplicate (field_search_id, nge_object_id) combinations in combined_results.")
if issues:
    print("Potential issues detected:")
    for i in issues:
        print(" -", i)
else:
    print("No major structural issues detected by this validator.")


=== SUMMARY FLAGS ===
Potential issues detected:
 - Some image filenames do not parse valid field_search_id/nge_object_id.


In [1]:
import os
import pandas as pd
from configparser import ConfigParser
from ConnectionPool import pool

excel_files = [f for f in os.listdir('.') if f.startswith('combined_results_') and f.endswith('.xlsx')]
if not excel_files:
    print("[ERROR] No combined_results_*.xlsx found in current directory.")
    exit(1)

excel_files.sort(key=lambda x: os.path.getmtime(x), reverse=True)
latest_excel = excel_files[0]
print(f"Using Excel: {latest_excel}")
combined_df = pd.read_excel(latest_excel)

required_cols = {
    'field_search_id',
    'nge_object_id',
    'predicted_sport'
}
missing_cols = required_cols - set(combined_df.columns)
if missing_cols:
    print(f"[ERROR] Missing required columns in Excel: {missing_cols}")
    exit(1)

# Optional: filter to confident predictions
if 'predicted_sport_type_probability' in combined_df.columns:
    combined_df = combined_df[
        combined_df['predicted_sport_type_probability'] >= 0.8
    ].copy()
    print(f"Filtered to confident rows: {len(combined_df)}")

['image_paths', 'model_paths', 'error', 'find_shift', 'database']
Connection pool created successfully
Using Excel: combined_results_2026-01-22T10-52-53.xlsx
Filtered to confident rows: 196


In [2]:
# 1. Fetch DB data for new_google_earth and nge_object
# -----------------------------

config = ConfigParser()
config.read('./config.ini')

def get_conn():
    try:
        conn = pool.getconn()
        print("DB connection OK.")
        return conn
    except Exception as e:
        print("DB connection FAILED:", e)
        return None

conn = get_conn()
if conn is None:
    exit(1)

try:
    cur = conn.cursor()

    # 1a) new_google_earth: map field_search_id -> preliminary sport
    # Adjust `object_sport` if your column name is different
    cur.execute("""
        SELECT field_search_id, object_sport
        FROM public.new_google_earth
    """)
    nge_rows = cur.fetchall()
    nge_df = pd.DataFrame(nge_rows, columns=['field_search_id', 'prelim_sport'])

    # 1b) nge_object: final object sports
    cur.execute("""
        SELECT nge_object_id, field_search_id, sport_name
        FROM public.nge_object
    """)
    obj_rows = cur.fetchall()
    obj_df = pd.DataFrame(obj_rows,
                          columns=['nge_object_id', 'field_search_id', 'final_sport'])

finally:
    cur.close()
    pool.putconn(conn)

print(f"new_google_earth rows: {len(nge_df)}")
print(f"nge_object rows: {len(obj_df)}")

DB connection OK.
new_google_earth rows: 101
nge_object rows: 232


In [3]:
# 2. Build a unified comparison table
# -----------------------------

# First join Excel with nge_object (exact pair)
merged = combined_df.merge(
    obj_df,
    on=['field_search_id', 'nge_object_id'],
    how='left'
)

# Then join preliminary sport by field_search_id
merged = merged.merge(
    nge_df,
    on='field_search_id',
    how='left'
)

print("\nUnified comparison sample:")
print(merged[['field_search_id',
              'nge_object_id',
              'prelim_sport',
              'predicted_sport',
              'final_sport']].head(10))

# Keep only rows where at least one DB label exists
valid = merged.copy()

# -----------------------------
# 3. Comparison metrics
# -----------------------------

def compute_agreement(df, col_a, col_b, label):
    subset = df[df[col_a].notna() & df[col_b].notna()].copy()
    if subset.empty:
        print(f"\n[{label}] No overlapping rows with both {col_a} and {col_b} present.")
        return

    subset['match'] = subset[col_a] == subset[col_b]
    acc = subset['match'].mean()
    print(f"\n[{label}] Rows compared: {len(subset)}")
    print(f"[{label}] Agreement between {col_a} and {col_b}: {acc:.3f}")

    mism = subset[~subset['match']]
    print(f"[{label}] Mismatches: {len(mism)}")
    if not mism.empty:
        print(mism[['field_search_id',
                    'nge_object_id',
                    col_a,
                    col_b]].head(10))

# 3a) Preliminary vs predicted
compute_agreement(valid, 'prelim_sport', 'predicted_sport',
                  label="Preliminary vs Predicted")

# 3b) Predicted vs final (nge_object)
compute_agreement(valid, 'predicted_sport', 'final_sport',
                  label="Predicted vs Final")

# 3c) Preliminary vs final
compute_agreement(valid, 'prelim_sport', 'final_sport',
                  label="Preliminary vs Final")

# -----------------------------
# 4. Per-field majority comparison (optional)
# -----------------------------

# Some fields have multiple nge_object rows; compare majority predicted sport per field
print("\n[Field-level majority comparison]")

if 'predicted_sport' in combined_df.columns:
    # majority predicted sport per field_search_id
    field_majority = (combined_df.groupby('field_search_id')['predicted_sport']
                                  .agg(lambda x: x.value_counts().idxmax())
                                  .reset_index(name='predicted_sport_majority'))

    # join with preliminary
    field_cmp = field_majority.merge(nge_df, on='field_search_id', how='left')

    # join with one representative final_sport per field (e.g., most frequent)
    final_per_field = (obj_df.groupby('field_search_id')['final_sport']
                              .agg(lambda x: x.value_counts().idxmax())
                              .reset_index())
    final_per_field.rename(columns={'final_sport': 'final_sport_majority'}, inplace=True)

    field_cmp = field_cmp.merge(final_per_field, on='field_search_id', how='left')

    print(field_cmp.head())

    # majority vs prelim
    compute_agreement(field_cmp, 'prelim_sport', 'predicted_sport_majority',
                      label="Field-level Preliminary vs PredictedMajority")

    # majority vs final
    compute_agreement(field_cmp, 'predicted_sport_majority', 'final_sport_majority',
                      label="Field-level PredictedMajority vs FinalMajority")

else:
    print("predicted_sport column is missing; cannot do field-level comparison.")


Unified comparison sample:
   field_search_id  nge_object_id prelim_sport predicted_sport final_sport
0             9732            789                       Soccer     Stadium
1             9732            905                       Soccer     Stadium
2             9738            791                    Buildings    Baseball
3             9738            792                       Soccer     Stadium
4             9738            907                    Buildings    Baseball
5             9738            908                       Soccer     Stadium
6             9739            793                       Soccer      Soccer
7             9739            909                       Soccer      Soccer
8             9740            794                       Tennis      Tennis
9             9740            796                       Tennis      Tennis

[Preliminary vs Predicted] Rows compared: 196
[Preliminary vs Predicted] Agreement between prelim_sport and predicted_sport: 0.000
[Preliminary vs

KeyError: "['nge_object_id'] not in index"

In [1]:
import os
import sys
import pandas as pd
from configparser import ConfigParser
from ConnectionPool import pool

# -----------------------------
# Config & paths
# -----------------------------

config = ConfigParser()
config.read('./config.ini')

small_img_pth = config['image_paths']['total_zoom_small_path']
large_img_pth = config['image_paths']['total_zoom_large_path']

# thresholds (tune as you collect data)
THRESHOLDS = {
    "pair_rate_min": 0.98,
    "high_conf_coverage_min": 0.6,
    "accuracy_obj_min": 0.8,
    "accuracy_field_min": 0.8,
    "max_class_share_max": 0.95,
}

metrics = {}
issues = []

# -----------------------------
# Helpers
# -----------------------------

def list_image_files(directory):
    if not os.path.isdir(directory):
        print(f"[WARN] Directory does not exist: {directory}")
        return []
    exts = ('.png', '.jpg', '.jpeg')
    return sorted([f for f in os.listdir(directory) if f.lower().endswith(exts)])

def parse_ids_from_filename(filename):
    """
    Expected: nge_object_id_field_search_id_sport_name.ext
    Returns (field_search_id, nge_object_id)
    """
    base = os.path.splitext(filename)[0]
    parts = base.split("_")
    nge_object_id = None
    field_search_id = None
    if len(parts) >= 2:
        try:
            nge_object_id = int(parts[0])
        except ValueError:
            pass
        try:
            field_search_id = int(parts[1])
        except ValueError:
            pass
    return field_search_id, nge_object_id

def get_conn():
    try:
        conn = pool.getconn()
        print("DB connection OK.")
        return conn
    except Exception as e:
        print("DB connection FAILED:", e)
        return None

# -----------------------------
# 1) Image integrity checks
# -----------------------------

print("=== IMAGE INTEGRITY CHECKS ===")
small_files = list_image_files(small_img_pth)
large_files = list_image_files(large_img_pth)

metrics["small_image_count"] = len(small_files)
metrics["large_image_count"] = len(large_files)

small_ids = set(os.path.splitext(f)[0] for f in small_files)
large_ids = set(os.path.splitext(f)[0] for f in large_files)

missing_in_large = small_ids - large_ids
missing_in_small = large_ids - small_ids

metrics["missing_large_count"] = len(missing_in_large)
metrics["missing_small_count"] = len(missing_in_small)

if small_files:
    metrics["pair_rate"] = 1.0 - (len(missing_in_large) / len(small_files))
else:
    metrics["pair_rate"] = 0.0

print(f"Small images: {metrics['small_image_count']}")
print(f"Large images: {metrics['large_image_count']}")
print(f"Missing in large: {metrics['missing_large_count']}")
print(f"Missing in small: {metrics['missing_small_count']}")
print(f"Pair rate: {metrics['pair_rate']:.3f}")

if metrics["pair_rate"] < THRESHOLDS["pair_rate_min"]:
    issues.append(f"Pair rate below threshold ({metrics['pair_rate']:.3f} < {THRESHOLDS['pair_rate_min']})")

# Filename parse check
parsed = []
bad_name = 0
for f in small_files:
    field_id, obj_id = parse_ids_from_filename(f)
    if field_id is None or obj_id is None:
        bad_name += 1
    parsed.append((f, field_id, obj_id))
metrics["invalid_image_names"] = bad_name
print(f"Invalid image filenames (cannot parse IDs): {bad_name}")
if bad_name > 0:
    issues.append("Some image filenames do not parse valid field_search_id/nge_object_id")

img_df = pd.DataFrame(parsed, columns=["filename", "field_search_id", "nge_object_id"])
dup_pairs = (img_df.groupby(["field_search_id", "nge_object_id"])
                  .size().reset_index(name="count"))
dup_pairs = dup_pairs[dup_pairs["count"] > 1]
metrics["duplicate_image_pairs"] = len(dup_pairs)
print(f"Duplicate (field_search_id, nge_object_id) image pairs: {metrics['duplicate_image_pairs']}")
if metrics["duplicate_image_pairs"] > 0:
    issues.append("Duplicate (field_search_id, nge_object_id) pairs in images")

# -----------------------------
# 2) Excel / model output sanity
# -----------------------------

print("\n=== EXCEL / MODEL OUTPUT CHECK ===")
excel_files = [f for f in os.listdir('.') if f.startswith('combined_results_') and f.endswith('.xlsx')]
if not excel_files:
    print("[ERROR] No combined_results_*.xlsx found.")
    issues.append("No combined_results Excel found")
    combined_df = pd.DataFrame()
else:
    excel_files.sort(key=lambda x: os.path.getmtime(x), reverse=True)
    latest_excel = excel_files[0]
    print(f"Using Excel: {latest_excel}")
    combined_df = pd.read_excel(latest_excel)

if combined_df.empty:
    metrics["excel_row_count"] = 0
    issues.append("combined_results Excel is empty")
else:
    metrics["excel_row_count"] = len(combined_df)
    print("Rows in combined_results:", metrics["excel_row_count"])

    prob_cols = [
        'predicted_field_type_probability',
        'predicted_sport_type_probability',
        'large_predicted_field_type_probability',
        'large_predicted_sport_type_probability'
    ]
    out_of_range = 0
    for col in prob_cols:
        if col in combined_df.columns:
            col_bad = ((combined_df[col] < 0) | (combined_df[col] > 1)).sum()
            out_of_range += col_bad
    metrics["out_of_range_probs"] = out_of_range
    print("Out-of-range probabilities:", out_of_range)
    if out_of_range > 0:
        issues.append("Some probability values are outside [0,1]")

    # Confidence coverage
    if 'predicted_sport_type_probability' in combined_df.columns:
        hc = (combined_df['predicted_sport_type_probability'] >= 0.8).mean()
        metrics["high_conf_coverage"] = float(hc)
    else:
        metrics["high_conf_coverage"] = 0.0
        issues.append("predicted_sport_type_probability missing in Excel")

    print(f"High-confidence coverage (p>=0.8): {metrics['high_conf_coverage']:.3f}")
    if metrics["high_conf_coverage"] < THRESHOLDS["high_conf_coverage_min"]:
        issues.append(
            f"High-confidence coverage below threshold ({metrics['high_conf_coverage']:.3f} < {THRESHOLDS['high_conf_coverage_min']})"
        )

    # Class distribution
    if 'predicted_sport' in combined_df.columns:
        class_dist = combined_df['predicted_sport'].value_counts(normalize=True)
        metrics["class_dist"] = class_dist.to_dict()
        max_share = class_dist.max()
        metrics["max_class_share"] = float(max_share)
        print("Predicted_sport distribution:", metrics["class_dist"])
        print(f"Max class share: {metrics['max_class_share']:.3f}")
        if metrics["max_class_share"] > THRESHOLDS["max_class_share_max"]:
            issues.append(
                f"One class dominates predictions (share={metrics['max_class_share']:.3f} > {THRESHOLDS['max_class_share_max']})"
            )
    else:
        metrics["class_dist"] = {}
        metrics["max_class_share"] = 0.0
        issues.append("predicted_sport missing in Excel")

    # Duplicate ID pairs in Excel
    if {"field_search_id", "nge_object_id"}.issubset(combined_df.columns):
        dup_excel = (combined_df.groupby(["field_search_id", "nge_object_id"])
                                 .size().reset_index(name="count"))
        dup_excel = dup_excel[dup_excel["count"] > 1]
        metrics["duplicate_excel_pairs"] = len(dup_excel)
        print("Duplicate (field_search_id, nge_object_id) in Excel:", metrics["duplicate_excel_pairs"])
        if metrics["duplicate_excel_pairs"] > 0:
            issues.append("Duplicate (field_search_id, nge_object_id) in combined_results Excel")

# -----------------------------
# 3) DB consistency & accuracy
# -----------------------------

print("\n=== DB CONSISTENCY & ACCURACY CHECK ===")
conn = get_conn()
if conn and not combined_df.empty:
    try:
        cur = conn.cursor()

        # basic counts
        cur.execute("SELECT COUNT(*) FROM public.new_google_earth;")
        metrics["new_google_earth_count"] = cur.fetchone()[0]
        cur.execute("SELECT COUNT(*) FROM public.nge_object;")
        metrics["nge_object_count"] = cur.fetchone()[0]
        print("Rows in new_google_earth:", metrics["new_google_earth_count"])
        print("Rows in nge_object:", metrics["nge_object_count"])

        # Predicted vs final (per-object)
        ids = combined_df[['field_search_id', 'nge_object_id']].dropna().drop_duplicates()
        if not ids.empty:
            placeholders = ",".join(["(%s,%s)"] * len(ids))
            params = []
            for _, r in ids.iterrows():
                params.extend([int(r['field_search_id']), int(r['nge_object_id'])])

            query = f"""
                SELECT no.nge_object_id,
                       no.field_search_id,
                       no.sport_name
                FROM public.nge_object no
                JOIN (VALUES {placeholders}) AS v(field_search_id, nge_object_id)
                  ON no.field_search_id = v.field_search_id
                 AND no.nge_object_id = v.nge_object_id;
            """
            cur.execute(query, tuple(params))
            db_rows = cur.fetchall()
            db_df = pd.DataFrame(db_rows,
                                 columns=['nge_object_id', 'field_search_id', 'final_sport'])

            merged = combined_df.merge(
                db_df,
                on=['field_search_id', 'nge_object_id'],
                how='inner'
            )
            if not merged.empty and 'predicted_sport' in merged.columns:
                valid = merged[merged['final_sport'].notna()].copy()
                if not valid.empty:
                    valid['correct'] = valid['predicted_sport'] == valid['final_sport']
                    acc = valid['correct'].mean()
                    metrics["accuracy_obj"] = float(acc)
                else:
                    metrics["accuracy_obj"] = 0.0
            else:
                metrics["accuracy_obj"] = 0.0
        else:
            metrics["accuracy_obj"] = 0.0

        print(f"Per-object predicted vs final accuracy: {metrics['accuracy_obj']:.3f}")
        if metrics["accuracy_obj"] < THRESHOLDS["accuracy_obj_min"]:
            issues.append(
                f"Per-object accuracy below threshold ({metrics['accuracy_obj']:.3f} < {THRESHOLDS['accuracy_obj_min']})"
            )

        # Field-level majority vs final majority
        if 'predicted_sport' in combined_df.columns:
            field_majority = (combined_df.groupby('field_search_id')['predicted_sport']
                              .agg(lambda x: x.value_counts().idxmax())
                              .reset_index(name='predicted_sport_majority'))

            cur.execute("""
                SELECT field_search_id, sport_name
                FROM public.nge_object;
            """)
            rows2 = cur.fetchall()
            obj2_df = pd.DataFrame(rows2, columns=['field_search_id', 'final_sport'])
            final_per_field = (obj2_df.groupby('field_search_id')['final_sport']
                               .agg(lambda x: x.value_counts().idxmax())
                               .reset_index(name='final_sport_majority'))

            field_cmp = field_majority.merge(final_per_field, on='field_search_id', how='inner')
            if not field_cmp.empty:
                field_cmp['correct'] = field_cmp['predicted_sport_majority'] == field_cmp['final_sport_majority']
                acc_field = field_cmp['correct'].mean()
                metrics["accuracy_field"] = float(acc_field)
            else:
                metrics["accuracy_field"] = 0.0
        else:
            metrics["accuracy_field"] = 0.0

        print(f"Field-level majority accuracy: {metrics['accuracy_field']:.3f}")
        if metrics["accuracy_field"] < THRESHOLDS["accuracy_field_min"]:
            issues.append(
                f"Field-level accuracy below threshold ({metrics['accuracy_field']:.3f} < {THRESHOLDS['accuracy_field_min']})"
            )

        # Duplicate pairs in DB
        cur.execute("""
            SELECT field_search_id, nge_object_id, COUNT(*)
            FROM public.nge_object
            GROUP BY field_search_id, nge_object_id
            HAVING COUNT(*) > 1;
        """)
        dup_db_rows = cur.fetchall()
        metrics["duplicate_db_pairs"] = len(dup_db_rows)
        print("Duplicate (field_search_id, nge_object_id) in DB:", metrics["duplicate_db_pairs"])
        if metrics["duplicate_db_pairs"] > 0:
            issues.append("Duplicate (field_search_id, nge_object_id) in nge_object table")

    except Exception as e:
        print("Error during DB validation:", e)
        issues.append("DB validation error")
    finally:
        cur.close()
        pool.putconn(conn)
else:
    if not conn:
        issues.append("No DB connection for validation")
    if combined_df.empty:
        issues.append("No Excel data for DB comparison")

# -----------------------------
# 4) Summary & exit code
# -----------------------------

print("\n=== PIPELINE HEALTH REPORT ===")
for k, v in metrics.items():
    if isinstance(v, dict):
        print(f"{k}: {v}")
    else:
        print(f"{k}: {v}")

if issues:
    print("\n[STATUS] FAIL / WARN")
    for i in issues:
        print(" -", i)
    sys.exit(1)
else:
    print("\n[STATUS] OK: No major issues detected.")
    sys.exit(0)


['image_paths', 'model_paths', 'error', 'find_shift', 'database']
Connection pool created successfully
=== IMAGE INTEGRITY CHECKS ===
Small images: 332
Large images: 332
Missing in large: 0
Missing in small: 0
Pair rate: 1.000
Invalid image filenames (cannot parse IDs): 332
Duplicate (field_search_id, nge_object_id) image pairs: 0

=== EXCEL / MODEL OUTPUT CHECK ===
Using Excel: combined_results_2026-01-22T10-52-53.xlsx
Rows in combined_results: 232
Out-of-range probabilities: 0
High-confidence coverage (p>=0.8): 0.845
Predicted_sport distribution: {'Tennis': 0.5431034482758621, 'Soccer': 0.19827586206896552, 'Buildings': 0.16379310344827586, 'Basketball': 0.05172413793103448, 'Baseball': 0.04310344827586207}
Max class share: 0.543
Duplicate (field_search_id, nge_object_id) in Excel: 0

=== DB CONSISTENCY & ACCURACY CHECK ===
DB connection OK.
Rows in new_google_earth: 101
Rows in nge_object: 232
Per-object predicted vs final accuracy: 0.638
Field-level majority accuracy: 0.467
Duplica

SystemExit: 1

c:\Users\owner\Documents\Gameplay_FieldAutomation_full_v3\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [1]:
start_validation(size='small', sample=50)
# or
start_validation(size='large', sample=50)


NameError: name 'start_validation' is not defined

In [4]:
import pandas as pd

df = pd.read_excel("C:\Users\owner\Documents\Gameplay_FieldAutomation_full_v3\Gameplay_FieldAutomation_full_v3\match_validation_small.xlsx")

def flag_row(row):
    # split comma-separated values, strip spaces
    items = {x.strip() for x in str(row["actual_sport"]).split(",") if x.strip()}
    refs = {str(row["detected_sport_type"]).strip(), str(row["predicted_sport"]).strip()}
    # 0 if any match, 1 if none
    return 0 if items & refs else 1

df["flag"] = df.apply(flag_row, axis=1)

# Save back to CSV
df.to_excel("match_validation_small_flagged.xlsx", index=False)


SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (2414599858.py, line 3)